# Transformer Fine-Tuning with Cross-Validation for Twitter Classification

**Objective**: Fine-tune a Twitter-XLM-RoBERTa model for binary classification (Influencer vs Observer) and ensemble with XGBoost on structured features.

## Pipeline Overview

1. **Data Preparation**: Load JSONL data, extract text + metadata, create pseudo user IDs
2. **Transformer Fine-Tuning**: StratifiedGroupKFold CV to prevent user leakage
3. **XGBoost Training**: Train on pre-computed structured features
4. **Ensemble**: Blend Transformer + XGBoost predictions with threshold optimization

## Key Design Choices

- **Model**: `cardiffnlp/twitter-xlm-roberta-base` (optimized for multilingual tweets)
- **CV Strategy**: StratifiedGroupKFold with pseudo user IDs to prevent data leakage
- **Mixed Precision**: FP16 training when GPU is available
- **Ensemble**: Simple weighted average with threshold calibration on OOF predictions

## 1. Environment Setup

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
import lightgbm as lgb

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch CUDA available: True
GPU name: NVIDIA RTX 4000 Ada Generation


In [2]:
# ============================================================================
# CACHE CONFIGURATION
# Redirect HuggingFace cache to /tmp to minimize disk writes
# ============================================================================

os.environ["HF_HOME"] = "/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf/datasets"
os.environ["HF_DATASETS_DOWNLOADED_DATASETS"] = "/tmp/hf/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Path(os.environ["TRANSFORMERS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================================
# REPRODUCIBILITY
# ============================================================================

SEED = 42

def set_seed(seed: int = 42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## 2. Configuration

In [4]:
# ============================================================================
# HYPERPARAMETERS AND PATHS
# ============================================================================

# Model configuration
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base"  # Optimized for multilingual tweets
MAX_LEN = 160
N_SPLITS = 4

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 16
GRAD_ACC = 2  # Effective batch size = 32
LR = 2e-5
WEIGHT_DECAY = 0.01

# Project paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # Handle notebooks/ subdirectory

DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submission"
MODEL_DIR = PROJECT_ROOT / "models/transformer_cv"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {PROJECT_ROOT}")

Working directory: /users/eleves-a/2023/malo.tamalet/influencer-or-observer-1


## 3. Data Processing Utilities

In [5]:
# ============================================================================
# TEXT AND METADATA EXTRACTION UTILITIES
# ============================================================================

def parse_source(source_html: str) -> str:
    """
    Parse tweet source HTML to identify posting device/client.
    Bot detection is important as automated accounts often correlate with influencers.
    """
    if not isinstance(source_html, str):
        return "unknown"
    src = source_html.lower()
    if "iphone" in src:
        return "iphone"
    if "android" in src:
        return "android"
    if "tweetdeck" in src:
        return "tweetdeck"
    if "web" in src or "browser" in src:
        return "web"
    if any(x in src for x in ["buffer", "hootsuite", "socialflow", "sprout", "dlvr.it"]):
        return "bot"
    return "other"


def extract_text(row: pd.Series) -> str:
    """Extract full tweet text, handling extended tweets and truncation."""
    if isinstance(row.get("extended_tweet"), dict):
        full = row["extended_tweet"].get("full_text")
        if full:
            return str(full)
    if isinstance(row.get("full_text"), str) and row["full_text"]:
        return str(row["full_text"])
    if isinstance(row.get("text"), str):
        return str(row["text"])
    return ""


def build_input_text(row: pd.Series) -> str:
    """
    Build model input by concatenating tweet text with user metadata.
    Format: <tweet_text>[DESC] <user_description>[META] <structured_features>
    """
    user = row.get("user") or {}
    text = extract_text(row)
    desc = (user.get("description") or "")[:160]
    location = (user.get("location") or "")[:80]
    source = parse_source(row.get("source", ""))
    
    # Extract available user statistics
    statuses = user.get("statuses_count") or 0
    favourites = user.get("favourites_count") or 0
    listed = user.get("listed_count") or 0
    has_url = 1 if user.get("url") else 0
    is_reply = 1 if row.get("in_reply_to_status_id") is not None else 0
    
    meta = f"device={source} statuses={statuses} favs={favourites} listed={listed} reply={is_reply} url={has_url} loc={location}"
    return f"{text}[DESC] {desc}[META] {meta}"


def pseudo_user_id(row: pd.Series) -> str:
    """
    Generate pseudo user ID from profile characteristics.
    Used for StratifiedGroupKFold to prevent user leakage across folds.
    Note: Real user IDs are not available in the dataset.
    """
    user = row.get("user") or {}
    key = "|".join([
        str(user.get("description", ""))[:64],
        str(user.get("profile_image_url_https", "")),
        str(user.get("profile_banner_url", "")),
        str(user.get("statuses_count", 0)),
    ])
    return str(abs(hash(key)) % (10 ** 12))

## 4. Data Loading

In [6]:
# ============================================================================
# LOAD AND PREPROCESS DATA
# ============================================================================

train_raw = pd.read_json(DATA_DIR / "train.jsonl", lines=True)
test_raw = pd.read_json(DATA_DIR / "kaggle_test.jsonl", lines=True)

# Build model inputs and pseudo user IDs
train_raw["text_input"] = train_raw.apply(build_input_text, axis=1)
test_raw["text_input"] = test_raw.apply(build_input_text, axis=1)
train_raw["group"] = train_raw.apply(pseudo_user_id, axis=1)
test_raw["group"] = test_raw.apply(pseudo_user_id, axis=1)

# Extract arrays for training
labels = train_raw["label"].astype(int).to_numpy()
groups = train_raw["group"].to_numpy()
test_ids = test_raw["challenge_id"].astype(int).to_numpy()

print("Label distribution:")
print(train_raw[["label"]].value_counts(normalize=True))
print(f"\nTrain shape: {train_raw.shape}, Test shape: {test_raw.shape}")

Label distribution:
label
0        0.533677
1        0.466323
Name: proportion, dtype: float64

Train shape: (154914, 39), Test shape: (103380, 37)


## 5. Tokenization

In [7]:
# ============================================================================
# TOKENIZATION WITH HUGGINGFACE DATASETS
# ============================================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_raw[["text_input", "label", "group"]])
test_ds = Dataset.from_pandas(test_raw[["challenge_id", "text_input", "group"]])

# Remove auxiliary columns (handle missing __index_level_0__ gracefully)
train_remove_cols = [c for c in ["text_input", "group", "__index_level_0__"] if c in train_ds.column_names]
test_remove_cols = [c for c in ["text_input", "group", "challenge_id", "__index_level_0__"] if c in test_ds.column_names]

def tokenize_fn(batch):
    return tokenizer(batch["text_input"], truncation=True, max_length=MAX_LEN)

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_remove_cols)
test_tokenized = test_ds.map(tokenize_fn, batched=True, remove_columns=test_remove_cols)

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

Map: 100%|██████████| 103380/103380 [00:16<00:00, 6444.58 examples/s]


## 6. Transformer Cross-Validation Training

In [ ]:
# ============================================================================
# TRANSFORMER FINE-TUNING WITH CROSS-VALIDATION
# ============================================================================

def compute_metrics(eval_pred):
    """Compute accuracy for evaluation."""
    logits, labels_arr = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels_arr, preds)}

# Initialize cross-validation with group awareness
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Initialize arrays for out-of-fold predictions
oof_probs = np.zeros(len(train_tokenized))
test_probs = np.zeros((len(test_tokenized), 2))

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels, groups)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_SPLITS}")
    print(f"{'='*60}")
    
    train_split = train_tokenized.select(train_idx.tolist())
    val_split = train_tokenized.select(val_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    out_dir = Path("/tmp/transformer_cv") / f"fold{fold}"
    out_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        gradient_accumulation_steps=GRAD_ACC,
        warmup_ratio=0.1,
        logging_strategy="no",
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=4,
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_split,
        eval_dataset=val_split,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Collect validation predictions
    val_logits = trainer.predict(val_split).predictions
    val_prob = softmax(val_logits, axis=1)[:, 1]
    oof_probs[val_idx] = val_prob

    # Accumulate test predictions (will be averaged later)
    test_logits = trainer.predict(test_tokenized).predictions
    test_probs += softmax(test_logits, axis=1)

# Average test predictions across folds
test_probs /= N_SPLITS

# Threshold optimization on out-of-fold predictions
thresholds = np.linspace(0.35, 0.65, 21)
best_thr = 0.5
best_acc = 0
for thr in thresholds:
    acc = accuracy_score(labels, (oof_probs >= thr).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thr = thr

print(f"\n{'='*60}")
print(f"TRANSFORMER RESULTS")
print(f"{'='*60}")
print(f"OOF Accuracy: {best_acc:.4f} @ threshold={best_thr:.3f}")
transformer_test_pred = (test_probs[:, 1] >= best_thr).astype(int)


FOLD 1/4


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_38107/228398861.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


## 7. XGBoost on Structured Features

In [ ]:
# ============================================================================
# XGBOOST TRAINING ON STRUCTURED FEATURES
# ============================================================================

from xgboost import XGBClassifier

# Average test predictions and optimize threshold (if resuming from above)
test_probs /= N_SPLITS

thresholds = np.linspace(0.35, 0.65, 21)
best_thr = 0.5
best_acc = 0
for thr in thresholds:
    acc = accuracy_score(labels, (oof_probs >= thr).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thr = thr

print(f"Transformer OOF Accuracy: {best_acc:.4f} @ threshold={best_thr:.3f}")
transformer_test_pred = (test_probs[:, 1] >= best_thr).astype(int)

# Load pre-computed structured features
X_feat = np.load(DATA_DIR / "features/X_train_features.npy")
X_test_feat = np.load(DATA_DIR / "features/X_kaggle_features.npy")

xgb_oof = np.zeros(len(labels))
xgb_test = np.zeros(len(X_test_feat))

print(f"\n{'='*60}")
print("XGBOOST CROSS-VALIDATION")
print(f"{'='*60}")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_feat, labels, groups)):
    print(f"Fold {fold+1}/{N_SPLITS}")
    
    clf = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=0.2,
        gamma=0.1,
        tree_method="hist",
        device="cuda",
        predictor="gpu_predictor",
        eval_metric="logloss",
        random_state=SEED,
        max_bin=512,
    )
    
    clf.fit(
        X_feat[tr_idx], labels[tr_idx],
        eval_set=[(X_feat[val_idx], labels[val_idx])],
        verbose=False
    )
    
    xgb_oof[val_idx] = clf.predict_proba(X_feat[val_idx])[:, 1]
    xgb_test += clf.predict_proba(X_test_feat)[:, 1]

xgb_test /= N_SPLITS
best_thr_xgb = max((accuracy_score(labels, (xgb_oof >= thr).astype(int)), thr) for thr in thresholds)[1]
print(f"\nXGBoost OOF Accuracy: {accuracy_score(labels, (xgb_oof >= best_thr_xgb).astype(int)):.4f} @ threshold={best_thr_xgb:.3f}")

## 8. Ensemble: Weighted Blending

In [ ]:
# ============================================================================
# WEIGHTED BLENDING: GRID SEARCH FOR OPTIMAL WEIGHT AND THRESHOLD
# ============================================================================

best_w, best_thr_blend, best_acc_blend = 0.6, best_thr, best_acc

for w in np.linspace(0, 1, 21):
    blended = w * oof_probs + (1 - w) * xgb_oof
    for thr in thresholds:
        acc = accuracy_score(labels, (blended >= thr).astype(int))
        if acc > best_acc_blend:
            best_acc_blend, best_w, best_thr_blend = acc, w, thr

print(f"Best Blend: weight={best_w:.2f} (Transformer), threshold={best_thr_blend:.3f}")
print(f"Blend OOF Accuracy: {best_acc_blend:.4f}")

# Generate blended predictions
final_proba = best_w * test_probs[:, 1] + (1 - best_w) * xgb_test
final_pred = (final_proba >= best_thr_blend).astype(int)

pd.DataFrame({"ID": test_ids, "Prediction": final_pred}).to_csv(
    SUBMISSION_DIR / "submission_transformer_blend.csv", index=False
)

## 9. Meta-Ensemble with Logistic Regression

In [ ]:
# ============================================================================
# META-ENSEMBLE: STACKING WITH LOGISTIC REGRESSION
# ============================================================================

from sklearn.linear_model import LogisticRegression

# Stack Transformer and XGBoost OOF predictions as features
stack_X = np.vstack([oof_probs, xgb_oof]).T
stack_test = np.vstack([test_probs[:, 1], xgb_test]).T

# Train meta-learner
meta = LogisticRegression(C=1.0, max_iter=1000)
meta.fit(stack_X, labels)

meta_oof = meta.predict_proba(stack_X)[:, 1]
meta_test = meta.predict_proba(stack_test)[:, 1]

# Optimize threshold on meta OOF predictions
best_thr_meta = 0.5
best_acc_meta = 0
for thr in thresholds:
    acc = accuracy_score(labels, (meta_oof >= thr).astype(int))
    if acc > best_acc_meta:
        best_acc_meta, best_thr_meta = acc, thr

print(f"{'='*60}")
print("META-ENSEMBLE RESULTS")
print(f"{'='*60}")
print(f"OOF Accuracy: {best_acc_meta:.4f} @ threshold={best_thr_meta:.3f}")

# Generate submission
final_pred = (meta_test >= best_thr_meta).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": final_pred}).to_csv(
    SUBMISSION_DIR / "submission_meta_lr.csv", index=False
)
print(f"Saved: {SUBMISSION_DIR / 'submission_meta_lr.csv'}")

## 10. Generate Submissions

In [ ]:
# ============================================================================
# GENERATE TRANSFORMER-ONLY SUBMISSIONS WITH VARIOUS THRESHOLDS
# ============================================================================

for thr in [0.55, 0.60, 0.65]:
    pred = (test_probs[:, 1] >= thr).astype(int)
    output_path = SUBMISSION_DIR / f"submission_transformer_thr_{thr:.2f}.csv"
    pd.DataFrame({"ID": test_ids, "Prediction": pred}).to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

In [ ]:
# ============================================================================
# FINAL SUBMISSION FILES
# ============================================================================

# Transformer-only submission
sub_transformer = pd.DataFrame({
    "ID": test_ids,
    "Prediction": transformer_test_pred.astype(int),
})
sub_transformer.to_csv(SUBMISSION_DIR / "submission_transformer_only.csv", index=False)

# Blended submission
sub_blend = pd.DataFrame({
    "ID": test_ids,
    "Prediction": final_pred.astype(int),
})
sub_blend.to_csv(SUBMISSION_DIR / "submission_transformer_blend.csv", index=False)

print(f"{'='*60}")
print("SUBMISSION FILES GENERATED")
print(f"{'='*60}")
print(f"  - {SUBMISSION_DIR / 'submission_transformer_only.csv'}")
print(f"  - {SUBMISSION_DIR / 'submission_transformer_blend.csv'}")
print(f"  - {SUBMISSION_DIR / 'submission_meta_lr.csv'}")